# Welcome to Week 4 - LangChain and LangGraph

You have built agents with the OpenAI Agents SDK, and you have built a crew with CrewAI. Now you get to meet the framework that started much of this movement, and the one you are most likely to see in production: LangChain, together with its orchestration engine LangGraph.

This week we take our time and work up from the ground floor, so that by Day 5 you can build something genuinely powerful with full understanding of what is happening underneath.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The LangChain docs</h2>
            <span style="color:#00bfff;">The 1.x documentation lives at <a href="https://docs.langchain.com/oss/python/">https://docs.langchain.com/oss/python/</a>. It is worth a browse. Watch out for older 0.x material that still floats around the web, because the modern API has moved on from it.
            </span>
        </td>
    </tr>
</table>

## The big idea: four layers

LangChain and LangGraph are best understood as four layers, where each layer is built on the one below it. As you climb, you hand more of the work to the framework and write less code yourself.

| Layer | Packages | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | chat models, the `@tool` decorator, messages, structured output | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state, memory and checkpointing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools and a prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | an opinionated harness with planning, sub-agents and a filesystem | your intent |

Two things to hold onto all week.

First, this is a stack you can tap at any altitude, not a ladder you climb and leave behind. Real applications mix the layers freely, and our Day 5 project does exactly that.

Second, the layers really are built on each other. The `create_agent` you meet on Day 3 is a LangGraph graph underneath, the very kind you will build by hand on Day 2. We start at the bottom so that this connection feels obvious when we reach it.

A quick word on the names. The langchain family covers both the Layer 1 building blocks and the Layer 3 agent, while langgraph in the middle is its own separate package. The rows above group them the clean way, one install per layer.

## Today is Layer 1: the building blocks

Think of Layer 1 as a well-stocked toolbox. It gives you a single interface to chat models from any provider, a simple way to define tools, a consistent set of message types, and structured output. If you have used LiteLLM, this is a heavier and more capable cousin.

We will cover, in order: calling a model, streaming, talking to other providers, messages, tools, giving tools to a model, running the tool loop by hand, and structured output.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Before you run this</h2>
            <span style="color:#ff7800;">This lab uses your OpenAI key, so make sure <code>OPENAI_API_KEY</code> is set in the <code>.env</code> file in your project root. One cell further down also reaches OpenRouter and needs <code>OPENROUTER_API_KEY</code>. If you do not have an OpenRouter account, you can simply skip that single cell, or point it at any other provider you do have.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# As always, our imports and environment variables come first, all in one place

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

### A first model call

`ChatOpenAI` is our window onto a model. We make one, then call `invoke` with a prompt. Back comes an `AIMessage`, and its `.content` holds the text.

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini")

reply = llm.invoke("In one sentence, what is an AI agent?")
print(reply.content)

### Streaming

For a live, token by token feel, swap `invoke` for `stream` and loop over the chunks. The model decides how to break the text into chunks, so the spacing can look uneven while it streams. The finished text reads normally.

In [ ]:
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

### Any OpenAI-compatible provider

Many providers speak the OpenAI dialect. To use one, point `base_url` at their endpoint and pass the right key. Here we reach a model on OpenRouter, which is a single gateway to hundreds of models. The model we ask for is Anthropic's Claude, so the very same `ChatOpenAI` class is now driving a completely different vendor's model. Nothing else about our code changes, which is the whole point of Layer 1.

In [ ]:
openrouter_llm = ChatOpenAI(
    model="anthropic/claude-haiku-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

reply = openrouter_llm.invoke("In one sentence, what is LangChain?")
print(reply.content)

### Messages

So far we passed a plain string, which is shorthand for a single user message. For real conversations we pass a list of messages. The common types are `SystemMessage`, `HumanMessage`, `AIMessage` and `ToolMessage`, and they all live in `langchain_core.messages`. You can also pass the plain role and content dictionaries you used with the OpenAI Agents SDK back in Week 2, whichever you prefer.

In [ ]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]

print(llm.invoke(messages).content)

# The exact same call using plain dictionaries, the format you already know
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of France?"},
]
print(llm.invoke(messages_as_dicts).content)

### Tools with the `@tool` decorator

A tool is a Python function the model is allowed to call. The modern way to make one is the `@tool` decorator. Your docstring becomes the description the model reads, and your type hints become the argument schema. This replaces the older `Tool(...)` wrapper from earlier versions of LangChain.

In [ ]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

### Giving tools to the model

Calling a tool yourself is fine, but we want the model to decide when to call it. We bind the tools to the model with `bind_tools`. Now when we invoke, the model may come back not with an answer but with a request to run a tool. That request shows up in `.tool_calls`.

In [ ]:
llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

### Running the tool loop by hand

Notice the model did not answer. It asked us to run `get_share_price`. So it falls to us to run the tool, hand the result back as a `ToolMessage`, and invoke again so the model can finish.

This little dance is the heart of every agent. The whole reason Layer 2 and Layer 3 exist is to do it for us. Run it once by hand here, and you will recognise it everywhere for the rest of the week.

We run a single round here because the model needs only one tool result to answer. A real agent wraps this in a loop that keeps going until the model stops asking for tools, which is exactly the job Layer 2 will take over.

Notice too that we look the tool up by name from a small dictionary, rather than hardcoding which function to call. With one tool that looks like overkill, but it is exactly what you need the moment there is more than one tool to choose from.

In [ ]:
# A lookup from tool name to tool, so this works no matter how many tools we have
tools_by_name = {get_share_price.name: get_share_price}

# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    tool_to_run = tools_by_name[call["name"]]
    result = tool_to_run.invoke(call["args"])
    conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

### Structured output

Often you want a model to fill in a Python object rather than write prose. You describe the shape with a Pydantic model and call `with_structured_output`. What comes back is a validated instance of your class, ready to use. Just as the tool earlier read its docstring and type hints, here the model reads each `Field` description to know what to put where.

In [ ]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

## That is Layer 1

In one lab you have used a unified model interface across two providers, streamed a response, worked with messages, defined a tool, watched a model ask for it, run the tool loop by hand, and pulled out structured output. These are the building blocks that everything else is made from.

Tomorrow we move up to Layer 2 and let LangGraph orchestrate these pieces into a graph, which is where memory and real workflows begin.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Write a second tool of your own, perhaps one that looks up a fictional weather report for a city. Bind both tools to the model and ask a question that needs both, then run the tool loop by hand until the model gives a final answer. You will appreciate all the more why we are about to let the framework take over this job.
            </span>
        </td>
    </tr>
</table>